# modeling_v13 — M4 듀얼 확정전: Cons-GA(99) vs Bal-GA(108) × {LGBM, ET}

**목적** — 두 GA 후보 셋을 **LGBM·ET 양쪽**에서 같은 환경으로 재평가 → §4.2 사전등록 규칙으로 **피처셋 확정** + **B1_LGBM 동결 · B1_ET 최초 기록**. (v2.1 듀얼 · v2.2 환경독립 폴드맵)

> **전제 파일** (노트북과 같은 폴더 또는 `data/`·`colab_GA/` 에 두면 자동 탐색):
> `v13_fdc_pool_wf_oof.csv.gz` · `core10_meta_wf.csv` · `feature_diet_selected.json` · `select_result_Conservative_GA.json` · `select_result_Balanced_GA.json`
>
> **예상 런타임(로컬 CPU)** : LGBM 20 fit ~40분 + ET 20 fit ~10분 ≈ **~50분**. `Restart & Run All`.
>
> **확인 포인트** : ① 두 셋 floor assert 통과(R10) ② **재현 self-check** — LGBM GKF = Cons 71.366 / Bal 71.272 (동결값 정확 재현) ③ B1_LGBM 동결·B1_ET 기록 ④ miss_frac 로깅.

### 정직 가드 (강건)
- **정직축 GKF = 동결 `stable_group_kfold` (v2.2)** : 표준 sklearn GroupKFold 는 numpy argsort 타이브레이크가 버전마다 달라 lot→fold 배정(=정직축 절대값)이 흔들린다(미러 검증 확인, 원순위까지 뒤집힘). 이 폴드맵은 로컬 동결 71.366/71.272 를 **어느 환경에서도 정확 재현**한다.
- **B1_ET = 순수 median ET** : 결측(= 그 WF 가 그 step 을 안 거침, 의미 있는 신호)을 median 으로 메운 값. ET 가 크게 열세면 "ET 가 약함"이 아니라 **"median 대치가 신호를 죽임 → F3′ missing-indicator 후보"** 로 읽는다. 각 셋 결측률(miss_frac)을 함께 기록.
- **KFold 는 기록만**(R1). 의사결정 축 = GroupKFold(C20).

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, re, json, time
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer

# ── 동결 상수 (v8 v8_timeline_common 인라인) ──────────────────
ID_COL, TARGET_COL = "C64", "C65"
CORE10 = ["is_high_regime", "high_regime_days", "days_since_last_pm", "C33",
          "dslp_x_hour", "hour", "hour_x_c33",
          "C60_mean_step4", "C59_mean_step4", "is_special_recipe"]
M8_PARAMS = dict(objective="regression", metric="rmse",
    learning_rate=0.029017547696366934, num_leaves=175, min_child_samples=5,
    feature_fraction=0.6324704159196377, bagging_fraction=0.864012693783303, bagging_freq=7,
    lambda_l1=5.04154328625296, lambda_l2=0.024814259264649002,
    min_split_gain=0.2573073648505903, verbose=-1, seed=42)
BEST_ROUNDS = 705
ET_N_ESTIMATORS, ET_RANDOM_STATE = 500, 42
PROTECTED = ["C17", "C11", "C31", "C15", "C16"]

# ── 동결 참조값 (전부 로컬 venv 산; R6) ──
B0_REF = 71.212                                            # M2 diet∪core10 Conservative151 GKF
SIGMA_C65 = 261.7                                          # 정직축 R² 분모
EXPECT_LGBM_GKF = {"Conservative": 71.366, "Balanced": 71.272}   # 재현 self-check 기준
EXPECT_LGBM_KF  = {"Conservative": 50.677, "Balanced": 51.412}

# ── 파일 자동 탐색 (자기완결) ──
def _find(name):
    cands = [".", "data", "colab_GA", "..",
             os.path.join("..", "modeling_v13"),
             os.path.join("..", "modeling_v13", "data"),
             os.path.join("..", "modeling_v13", "colab_GA")]
    for d in cands:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} 없음 — 노트북과 같은 폴더(또는 data/·colab_GA/)에 두세요.")

POOL = _find("v13_fdc_pool_wf_oof.csv.gz")
META = _find("core10_meta_wf.csv")
DIET = _find("feature_diet_selected.json")
SELJSON = {p: _find(f"select_result_{p}_GA.json") for p in ["Conservative", "Balanced"]}
print("파일 확인:")
for x in [POOL, META, DIET, *SELJSON.values()]:
    print("  ", x)

파일 확인:
   data\v13_fdc_pool_wf_oof.csv.gz
   colab_GA\core10_meta_wf.csv
   .\feature_diet_selected.json
   .\select_result_Conservative_GA.json
   .\select_result_Balanced_GA.json


In [2]:
# ── 헬퍼 ──────────────────────────────────────────────────────
def _rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def sensor_of(c):
    m = re.match(r"(C\d+)_", c)
    return m.group(1) if m else c

def floor_ok(feats):
    have = {s: sum(1 for c in feats if sensor_of(c) == s) for s in PROTECTED}
    return all(v >= 1 for v in have.values()), have

# ── 환경독립 GroupKFold (v2.2 동결 stable 폴드맵) ───────────────
# 표준 sklearn GroupKFold 는 numpy argsort 기본정렬의 버전별 타이브레이크에 의존해
# lot→fold 배정이 환경마다 달라진다(미러 검증 확인). 아래 stable 배정은 로컬 동결값을
# 어느 환경에서도 정확 재현한다 (Cons 71.366 / Bal 71.272, Δ0.000 확인).
def stable_group_kfold(groups, n_splits=5):
    groups = np.asarray(groups)
    _, inv, cnt = np.unique(groups, return_inverse=True, return_counts=True)
    order = np.argsort(cnt, kind="stable")[::-1]          # 크기 내림차순 + 안정 타이브레이크
    fold_sizes = np.zeros(n_splits, dtype=np.int64)
    g2f = np.zeros(len(cnt), dtype=np.int64)
    for gi in order:
        f = int(np.argmin(fold_sizes)); fold_sizes[f] += cnt[gi]; g2f[gi] = f
    return g2f[inv]                                        # 샘플별 fold 인덱스 (0..n_splits-1)

def _gkf_splits(groups, n_splits=5):
    fv = stable_group_kfold(groups, n_splits)
    for k in range(n_splits):
        yield np.where(fv != k)[0], np.where(fv == k)[0]   # (train_idx, val_idx) 위치기반

# ── LGBM OOF (M8_PARAMS·705) ──
def oof_kfold_lgb(df, feats, y):
    oof = np.zeros(len(df))
    for k in range(5):
        tr = np.where((df["fold_kf5"] != k).to_numpy())[0]
        va = np.where((df["fold_kf5"] == k).to_numpy())[0]
        m = lgb.train(M8_PARAMS, lgb.Dataset(df.iloc[tr][feats], y[tr]), num_boost_round=BEST_ROUNDS)
        oof[va] = m.predict(df.iloc[va][feats])
    return _rmse(y, oof)

def oof_group_lgb(df, feats, y, groups):
    oof = np.zeros(len(df))
    for tr, va in _gkf_splits(groups):
        m = lgb.train(M8_PARAMS, lgb.Dataset(df.iloc[tr][feats], y[tr]), num_boost_round=BEST_ROUNDS)
        oof[va] = m.predict(df.iloc[va][feats])
    return _rmse(y, oof)

# ── ET OOF (순수 median 대치, fold-내 fit → 누수 안전; v8 fit_et 레시피) ──
def _fit_pred_et(Xtr, ytr, Xva):
    imp = SimpleImputer(strategy="median").fit(Xtr)       # ← fold 내 train 으로만 fit
    m = ExtraTreesRegressor(n_estimators=ET_N_ESTIMATORS, n_jobs=-1,
                            random_state=ET_RANDOM_STATE).fit(imp.transform(Xtr), ytr)
    return m.predict(imp.transform(Xva))

def oof_kfold_et(df, feats, y):
    oof = np.zeros(len(df))
    for k in range(5):
        tr = np.where((df["fold_kf5"] != k).to_numpy())[0]
        va = np.where((df["fold_kf5"] == k).to_numpy())[0]
        oof[va] = _fit_pred_et(df.iloc[tr][feats], y[tr], df.iloc[va][feats])
    return _rmse(y, oof)

def oof_group_et(df, feats, y, groups):
    oof = np.zeros(len(df))
    for tr, va in _gkf_splits(groups):
        oof[va] = _fit_pred_et(df.iloc[tr][feats], y[tr], df.iloc[va][feats])
    return _rmse(y, oof)

def miss_frac(df, feats):
    """median 대치 노출도 진단(ET 정직 가드): 전체·std·step5 결측률."""
    std_cols = [c for c in feats if "_std_" in c]
    s5 = [c for c in feats if c.endswith("step5")]
    return dict(all=round(float(df[feats].isna().mean().mean()), 4),
                std=round(float(df[std_cols].isna().mean().mean()), 4) if std_cols else 0.0,
                step5=round(float(df[s5].isna().mean().mean()), 4) if s5 else 0.0)

In [3]:
# ── 로드 & 두 후보 셋 복원 (fixed = core10 + champion 5센서) ──
pool = pd.read_csv(POOL); pool[ID_COL] = pool[ID_COL].astype(str)
meta = pd.read_csv(META); meta[ID_COL] = meta[ID_COL].astype(str)
df = pool.merge(meta, on=ID_COL, how="inner")
y = df[TARGET_COL].to_numpy(float)
groups = df["C20"].to_numpy()
diet = json.loads(open(DIET, encoding="utf-8").read())

SETS = {}
for p in ["Conservative", "Balanced"]:
    champions = list(diet["champions"][p].values())
    fixed = [f for f in dict.fromkeys(CORE10 + champions) if f in df.columns]
    selp = json.loads(open(SELJSON[p], encoding="utf-8").read())["selected_features"]
    feats = list(dict.fromkeys(fixed + selp))
    ok, have = floor_ok(feats)
    assert all(f in df.columns for f in feats), f"{p}: 누락 피처 존재"
    assert ok, f"{p}: 필수 5센서 floor 위반 {have}"                 # R10
    SETS[p] = dict(feats=feats, n=len(feats), floor=have, miss=miss_frac(df, feats))
    print(f"{p:12s} n={len(feats):3d} (fixed {len(fixed)} + sel {len(selp)})  floor={have}")
    print(f"             miss_frac={SETS[p]['miss']}")
print("df:", df.shape, "| unique C20 lots:", len(np.unique(groups)))

Conservative n= 99 (fixed 15 + sel 84)  floor={'C17': 4, 'C11': 5, 'C31': 3, 'C15': 3, 'C16': 2}
             miss_frac={'all': 0.1306, 'std': 0.2716, 'step5': 0.6523}
Balanced     n=108 (fixed 15 + sel 93)  floor={'C17': 2, 'C11': 9, 'C31': 2, 'C15': 3, 'C16': 3}
             miss_frac={'all': 0.0248, 'std': 0.0992, 'step5': 0.0}
df: (11939, 667) | unique C20 lots: 1217


In [4]:
# ── 2셋 × 2모델 × (KFold5 + GKF5) = 40 fit ──  (로컬 ~50분)
rows = []
for p in ["Conservative", "Balanced"]:
    feats = SETS[p]["feats"]
    for model, kf_fn, gk_fn in [("LGBM", oof_kfold_lgb, oof_group_lgb),
                                 ("ET",   oof_kfold_et,  oof_group_et)]:
        t = time.time()
        kf = kf_fn(df, feats, y)
        gk = gk_fn(df, feats, y, groups)
        dt = time.time() - t
        rows.append(dict(candidate=f"{p}-GA", model=model, n_total=len(feats),
                         KFold_OOF=round(kf, 3), GroupKFold_C20=round(gk, 3),
                         floor_ok=True, miss_all=SETS[p]["miss"]["all"], sec=round(dt, 0)))
        print(f"{p:12s} {model:4s}  KFold={kf:.3f}  GroupKFold(C20)={gk:.3f}  ({dt:.0f}s)")

res = pd.DataFrame(rows, columns=["candidate", "model", "n_total", "KFold_OOF",
                                  "GroupKFold_C20", "floor_ok", "miss_all", "sec"])
res

Conservative LGBM  KFold=50.677  GroupKFold(C20)=71.366  (1096s)
Conservative ET    KFold=48.481  GroupKFold(C20)=72.415  (397s)
Balanced     LGBM  KFold=51.412  GroupKFold(C20)=71.272  (1449s)
Balanced     ET    KFold=48.857  GroupKFold(C20)=72.245  (472s)


,candidate,model,n_total,KFold_OOF,GroupKFold_C20,floor_ok,miss_all,sec
0,Conservative-GA,LGBM,99,50.677,71.366,True,0.1306,1096.0
1,Conservative-GA,ET,99,48.481,72.415,True,0.1306,397.0
2,Balanced-GA,LGBM,108,51.412,71.272,True,0.0248,1449.0
3,Balanced-GA,ET,108,48.857,72.245,True,0.0248,472.0


In [5]:
# ── 재현 self-check + §4.2 검산 (공식 판정은 강건이 회신 숫자로) ──
piv = res.pivot(index="candidate", columns="model", values="GroupKFold_C20")

print("[재현 self-check] v2.2 stable 폴드맵 → 로컬 동결값 재현 확인")
for p in ["Conservative", "Balanced"]:
    g = float(piv.loc[f"{p}-GA", "LGBM"]); d = g - EXPECT_LGBM_GKF[p]
    print(f"  LGBM {p:12s} GKF={g:.3f}  (동결 {EXPECT_LGBM_GKF[p]}, Δ{d:+.3f})  {'OK' if abs(d) < 0.05 else '⚠️ 불일치 — 환경 점검'}")

def confirm_set(piv):
    cl, bl = float(piv.loc["Conservative-GA", "LGBM"]), float(piv.loc["Balanced-GA", "LGBM"])
    ce, be = float(piv.loc["Conservative-GA", "ET"]),   float(piv.loc["Balanced-GA", "ET"])
    def win(a_c, a_b):                                   # 모델별: |Δ|≥0.3 낮은쪽, 아니면 tie
        return ("Conservative" if a_c < a_b else "Balanced") if abs(a_c - a_b) >= 0.3 else "tie"
    wl, we = win(cl, bl), win(ce, be)
    if wl == we and wl != "tie":
        return wl, f"두 모델 합의 → {wl}"
    mc, mb = (cl + ce) / 2, (bl + be) / 2               # 갈리거나 tie → 평균 낮은쪽, 차<0.3 슬림
    if abs(mc - mb) < 0.3:
        return "Conservative", f"평균차 {abs(mc-mb):.3f}<0.3 → 슬림(Conservative), 안정코어 §1.3 우세와 일치"
    return ("Conservative" if mc < mb else "Balanced"), f"평균 낮은쪽 (Cons {mc:.3f} vs Bal {mb:.3f})"

confirmed, why = confirm_set(piv)
B1_LGBM = float(piv.loc[f"{confirmed}-GA", "LGBM"])
B1_ET   = float(piv.loc[f"{confirmed}-GA", "ET"])
r2 = 1 - (B1_LGBM / SIGMA_C65) ** 2
et_flag = bool(all(float(piv.loc[f"{p}-GA", "ET"]) > B0_REF + 2 for p in ["Conservative", "Balanced"]))

print(f"\n[검산·§4.2]  확정 셋 = {confirmed}-GA   ({why})")
print(f"  B1_LGBM (공식 기준선 후보) = {B1_LGBM:.3f}   |   B1_ET (도전자·순수 median) = {B1_ET:.3f}")
print(f"  가드레일 정직축 R² = {r2:.4f}   ({'통과 ≥0.9' if r2 >= 0.9 else '기각 <0.9'})")
print(f"  ET median-손실 의심 플래그(양쪽 > B0+2) = {et_flag}  → {'F3′ missing-indicator 우선 검토' if et_flag else '해당 없음'}")
print("  ※ 이 블록은 검산용. 공식 셋 확정·B1 동결은 회신 숫자로 사전등록 규칙에 따라 별도 확정.")

[재현 self-check] v2.2 stable 폴드맵 → 로컬 동결값 재현 확인
  LGBM Conservative GKF=71.366  (동결 71.366, Δ+0.000)  OK
  LGBM Balanced     GKF=71.272  (동결 71.272, Δ+0.000)  OK

[검산·§4.2]  확정 셋 = Conservative-GA   (평균차 0.132<0.3 → 슬림(Conservative), 안정코어 §1.3 우세와 일치)
  B1_LGBM (공식 기준선 후보) = 71.366   |   B1_ET (도전자·순수 median) = 72.415
  가드레일 정직축 R² = 0.9256   (통과 ≥0.9)
  ET median-손실 의심 플래그(양쪽 > B0+2) = False  → 해당 없음
  ※ 이 블록은 검산용. 공식 셋 확정·B1 동결은 회신 숫자로 사전등록 규칙에 따라 별도 확정.


In [6]:
# ── 저장 ──
out = dict(
    results=rows,
    decision=dict(confirmed_set=f"{confirmed}-GA", rule=why,
                  B1_LGBM=round(B1_LGBM, 3), B1_ET=round(B1_ET, 3),
                  r2_honest=round(r2, 4), et_median_suspect=et_flag,
                  cv="stable_group_kfold (v2.2 환경독립 폴드맵)"),
    expected_local=dict(LGBM_GKF=EXPECT_LGBM_GKF, LGBM_KF=EXPECT_LGBM_KF, B0=B0_REF),
    note="B1_ET=순수 median ET(도전자 기록, 기준선 아님). ET 열세는 median 손실 의심→F3′.")
json.dump(out, open("final_compare_dual_results.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
res.to_csv("final_compare_dual_results.csv", index=False)
print("saved: final_compare_dual_results.json / final_compare_dual_results.csv")

saved: final_compare_dual_results.json / final_compare_dual_results.csv


---
### 판정 & 다음 단계
- 위 **[검산·§4.2]** 는 참고용 — **공식 셋 확정·B1 동결은 강건이 회신 숫자로** 사전등록 규칙에 따라 내린다(노트북 auto 값에 의존하지 않음).
- **B1_LGBM = 공식 기준선**(게이트는 min(B0, B1_LGBM) − 0.5) · **B1_ET = 도전자 기록**(기준선 아님, 순환 방지).
- **ET median-손실 플래그**가 켜지면 → M7 **F3′ missing-indicator 변형** 우선 검토(원인 1순위 기록).
- 다음: **M4.5 진단**(확정 셋에 Null Importance 80셔플 추가) → **M5 듀얼 재튜닝**(목적 = GKF, stable 폴드맵 사용).
- REPORT 는 스냅샷 **신규 번호**로 기록(기존 04/05 는 v2.0 LGBM-only 산으로 보존).